In [4]:
!python3 -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in
%pip install python-woc pandas matplotlib tqdm

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 10.8 MB/s eta 0:00:0000:010:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 43.4 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 44.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.4/199.4 kB 21.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 27.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 26.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.2/355.2 kB 28.8 MB/s eta 0:00:00

In [10]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    "mobvoi_wenet",
    "pbs-assess_gfsynopsis",
    "pmlmodelling_ersem",
    "vearch_vearch",
    "hannorein_rebound",
    "soedinglab_mmseqs2",
    "hpddm_hpddm",
    "stephenslab_mr.mash.alpha",
    "gem_oq-engine",
    "sixin-zh_kymatio_wph",
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00068789555dbe4c6aff671288cb40ead8bf9105,mobvoi_wenet


In [28]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks

# chunks = [df['sha1'][x:x+40] for x in range(0, len(df), 40)]
commit_data = []

done = {item["commit"] for item in commit_data}
remaining = df.loc[~df["sha1"].isin(done), "sha1"]

chunks = [
    remaining.iloc[x:x+40]
    for x in range(0, len(remaining), 40)
]

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(4)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(
    df[["sha1", "project"]], left_on="commit", right_on="sha1"
)
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  5%|▌         | 297/5427 [20:51<5:51:35,  4.11s/it]

Got Errors {'0a1b5bc7422715af441f6f3f825fddb4d078040c': 'Key 0a1b5bc7422715af441f6f3f825fddb4d078040c not found in /da5_fast/All.sha1c/commit_10.tch'}


  8%|▊         | 436/5427 [30:23<5:41:08,  4.10s/it]

Got Errors {'e302bf378a7c58ff716bcbd8b947d7e09a4d1696': 'Key e302bf378a7c58ff716bcbd8b947d7e09a4d1696 not found in /da5_fast/All.sha1c/commit_99.tch'}


  9%|▊         | 468/5427 [32:34<5:39:08,  4.10s/it]

...

Got Errors {'f1fad75b4ace513580d10c655340aed5c94e5cfd': 'Key f1fad75b4ace513580d10c655340aed5c94e5cfd not found in /da5_fast/All.sha1c/commit_113.tch'}


 98%|█████████▊| 5331/5427 [6:05:27<06:33,  4.10s/it]

Got Errors {'fba2df9ec88cdeecda163012eb6b8f6adcd4dc30': 'Key fba2df9ec88cdeecda163012eb6b8f6adcd4dc30 not found in /da5_fast/All.sha1c/commit_123.tch'}


100%|██████████| 5427/5427 [6:12:01<00:00,  4.11s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00068789555dbe4c6aff671288cb40ead8bf9105,2331e07c0d80174bdbd36457128e47d9f79edcd6,[87886a897ec6c7da0fee38cf8ba266a52c76dc65],Binbin Zhang <binbzha@qq.com>,1651152697,+0800,Binbin Zhang <binbzha@qq.com>,1651152697,+0800,[binding]: add python binding\n,00068789555dbe4c6aff671288cb40ead8bf9105,mobvoi_wenet
1,000c7af5c36afafd3fa3ae4599af237e902c67bb,634e826cace15c002e2d0820a9a961af4e8111b5,[980507ecaba296a363ddb9a1cd8f93be6b2e8b50],Mddct <hamddct@gmail.com>,1697436927,+0800,Mddct <hamddct@gmail.com>,1697785167,+0800,[cli/paraformer] ali-paraformer load and inf...,000c7af5c36afafd3fa3ae4599af237e902c67bb,mobvoi_wenet


In [23]:
import httpx

try:
    response = httpx.get(
        "https://worldofcode.org/api/",
        timeout=20
    )
    print("Connected. HTTP status:", response.status_code)
except httpx.RequestError as error:
    print(type(error).__name__, str(error))
    print("Underlying cause:", repr(error.__cause__))

Connected. HTTP status: 404


In [7]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00144f85329256bf24124729a0554f76fda95877', 'tree': '116d1fe9034b233f623863ccaac7f9b597013d68', 'parent': ['f254d1c0a5e9e80b9f7e8d3b4709b9e93d52f618'], 'author': 'mmorise <mmorise@yamanashi.ac.jp>', 'author_time': 1492872352, 'author_tz': '+0900', 'committer': 'mmorise <mmorise@yamanashi.ac.jp>', 'committer_time': 1492872352, 'committer_tz': '+0900', 'message': 'DC component is removed\n'}


In [8]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [9]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,mmorise_world,00144f85329256bf24124729a0554f76fda95877,mmorise <mmorise@yamanashi.ac.jp>,1492872352,DC component is removed\n


In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='your_net_id_here'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

## GitHub project information

Current observations; each row records its collection time in UTC. Last commit uses the default branch's latest listed commit and its committer timestamp. This is not the historical September 30, 2025 status assessment.

| Project | Stars | Forks | Last commit (UTC) | Branch | Checked (UTC) | Repository | Status |
|---|---:|---:|---|---|---|---|---|
| mobvoi_wenet | 5243 | 1190 | 2026-09-07T07:21:08Z | main | 2026-10-08T03:42:48.905413+00:00 | https://github.com/wenet-e2e/wenet | ok |
| pbs-assess_gfsynopsis | 17 | 1 | 2026-09-19T23:19:56Z | master | 2026-10-08T03:42:51.508621+00:00 | https://github.com/pbs-assess/gfsynopsis | ok |
| pmlmodelling_ersem | 30 | 14 | 2026-08-18T13:51:19Z | master | 2026-10-08T03:42:54.094392+00:00 | https://github.com/pmlmodelling/ersem | ok |
| vearch_vearch | 2331 | 365 | 2026-07-27T05:29:53Z | master | 2026-10-08T03:42:56.587535+00:00 | https://github.com/vearch/vearch | ok |
| hannorein_rebound | 1138 | 270 | 2026-10-06T12:13:52Z | main | 2026-10-08T03:42:59.146263+00:00 | https://github.com/hannorein/rebound | ok |
| soedinglab_mmseqs2 | 2148 | 303 | 2026-09-30T04:36:10Z | master | 2026-10-08T03:43:01.645441+00:00 | https://github.com/soedinglab/MMseqs2 | ok |
| hpddm_hpddm | 160 | 37 | 2026-09-26T19:46:30Z | main | 2026-10-08T03:43:04.196691+00:00 | https://github.com/hpddm/hpddm | ok |
| stephenslab_mr.mash.alpha | 7 | 5 | 2026-02-16T16:00:01Z | master | 2026-10-08T03:43:07.010691+00:00 | https://github.com/stephenslab/mr.mashr | ok |
| gem_oq-engine | 451 | 325 | 2026-10-08T02:09:51Z | master | 2026-10-08T03:43:09.478557+00:00 | https://github.com/gem/oq-engine | ok |
| sixin-zh_kymatio_wph | 0 | 0 | 2025-04-10T14:37:11Z | master | 2026-10-08T03:43:12.093200+00:00 | https://github.com/sixin-zh/kymatio_wph | ok |
